<a href="https://colab.research.google.com/github/sangpm2005/BTH9-10_LTW/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import copy
import math
import random
import numpy as np   # Dùng numpy để in bàn cờ đẹp

# Các hằng số cho người chơi
X = "X"
O = "O"
EMPTY = None

# Sẽ được gán khi người chơi nhập
user = None
ai = None


# -----------------------------------------------
# TRẠNG THÁI BAN ĐẦU
# -----------------------------------------------
def initial_state():
    """
    Trả về trạng thái khởi tạo của bàn cờ (3x3).
    """
    return [
        [EMPTY, EMPTY, EMPTY],
        [EMPTY, EMPTY, EMPTY],
        [EMPTY, EMPTY, EMPTY]
    ]


# -----------------------------------------------
# XÁC ĐỊNH NGƯỜI CHƠI TIẾP THEO
# -----------------------------------------------
def player(board):
    """
    Trả về người chơi tiếp theo dựa trên số ô đã đánh.
    """
    count = 0
    for row in board:
        for cell in row:
            if cell is not None:
                count += 1

    # Nếu số ô đã đánh là số lẻ → tới lượt AI
    if count % 2 != 0:
        return ai
    return user


# -----------------------------------------------
# CÁC NƯỚC ĐI CÓ THỂ ĐI
# -----------------------------------------------
def actions(board):
    """
    Trả về tập các hành động (i, j) có thể đánh.
    """
    moves = set()
    for i in range(3):
        for j in range(3):
            if board[i][j] == EMPTY:
                moves.add((i, j))
    return moves


# -----------------------------------------------
# TRẢ VỀ BÀN CỜ MỚI SAU KHI ĐÁNH MỘT NƯỚC
# -----------------------------------------------
def result(board, action):
    """
    Trả về bàn cờ sau khi người chơi đánh vào ô action.
    """
    i, j = action
    new_board = copy.deepcopy(board)
    new_board[i][j] = player(board)
    return new_board


# -----------------------------------------------
# KIỂM TRA THẮNG THEO HÀNG NGANG
# -----------------------------------------------
def get_horizontal_winner(board):
    for row in board:
        if row[0] is not None and row.count(row[0]) == 3:
            return row[0]
    return None


# -----------------------------------------------
# KIỂM TRA THẮNG THEO CỘT
# -----------------------------------------------
def get_vertical_winner(board):
    for col in range(3):
        if (board[0][col] is not None and
            board[0][col] == board[1][col] == board[2][col]):
            return board[0][col]
    return None


# -----------------------------------------------
# KIỂM TRA THẮNG THEO ĐƯỜNG CHÉO
# -----------------------------------------------
def get_diagonal_winner(board):
    # Chéo trái sang phải
    if (board[0][0] is not None and
        board[0][0] == board[1][1] == board[2][2]):
        return board[0][0]

    # Chéo phải sang trái
    if (board[0][2] is not None and
        board[0][2] == board[1][1] == board[2][0]):
        return board[0][2]

    return None


# -----------------------------------------------
# HÀM TRẢ VỀ NGƯỜI THẮNG
# -----------------------------------------------
def winner(board):
    return (
        get_horizontal_winner(board)
        or get_vertical_winner(board)
        or get_diagonal_winner(board)
        or None
    )


# -----------------------------------------------
# KIỂM TRA GAME KẾT THÚC
# -----------------------------------------------
def terminal(board):
    if winner(board) is not None:
        return True

    for row in board:
        for cell in row:
            if cell is None:
                return False
    return True


# -----------------------------------------------
# GIÁ TRỊ UTILITY
# -----------------------------------------------
def utility(board):
    win = winner(board)
    if win == X:
        return 1
    if win == O:
        return -1
    return 0


# -----------------------------------------------
# MINIMAX – MAX NODE
# -----------------------------------------------
def maxValue(state):
    if terminal(state):
        return utility(state)

    v = -math.inf
    for action in actions(state):
        v = max(v, minValue(result(state, action)))
    return v


# -----------------------------------------------
# MINIMAX – MIN NODE
# -----------------------------------------------
def minValue(state):
    if terminal(state):
        return utility(state)

    v = math.inf
    for action in actions(state):
        v = min(v, maxValue(result(state, action)))
    return v


# -----------------------------------------------
# CHỌN NƯỚC TỐI ƯU
# -----------------------------------------------
def minimax(board):
    current = player(board)

    if current == X:
        best_value = -math.inf
        best_move = None
        for action in actions(board):
            check = minValue(result(board, action))
            if check > best_value:
                best_value = check
                best_move = action
    else:
        best_value = math.inf
        best_move = None
        for action in actions(board):
            check = maxValue(result(board, action))
            if check < best_value:
                best_value = check
                best_move = action

    return best_move


# -----------------------------------------------
# CHẠY GAME
# -----------------------------------------------
if __name__ == "__main__":
    board = initial_state()
    ai_turn = False

    print("Choose your player (X or O):")
    user = input().strip().upper()

    if user == X:
        ai = O
    else:
        ai = X

    while True:
        print("\nCurrent board:")
        print(np.array(board))

        if terminal(board):
            result_winner = winner(board)
            if result_winner is None:
                print("Game Over: Tie.")
            else:
                print(f"Game Over: {result_winner} wins!")
            break

        current = player(board)

        # Lượt AI
        if current == ai:
            print("\nAI is thinking...")
            move = minimax(board)
            board = result(board, move)

        # Lượt người chơi
        else:
            print("\nYour turn!")
            i = int(input("Row (0-2): "))
            j = int(input("Col (0-2): "))

            if board[i][j] == EMPTY:
                board = result(board, (i, j))
            else:
                print("Invalid move! Try again.")


Choose your player (X or O):
X

Current board:
[[None None None]
 [None None None]
 [None None None]]

Your turn!
Row (0-2): 0
Col (0-2): 0

Current board:
[['X' None None]
 [None None None]
 [None None None]]

AI is thinking...

Current board:
[['X' None None]
 [None 'O' None]
 [None None None]]

Your turn!
Row (0-2): 2
Col (0-2): 2

Current board:
[['X' None None]
 [None 'O' None]
 [None None 'X']]

AI is thinking...

Current board:
[['X' 'O' None]
 [None 'O' None]
 [None None 'X']]

Your turn!
Row (0-2): 2
Col (0-2): 1

Current board:
[['X' 'O' None]
 [None 'O' None]
 [None 'X' 'X']]

AI is thinking...

Current board:
[['X' 'O' None]
 [None 'O' None]
 ['O' 'X' 'X']]

Your turn!
Row (0-2): 0
Col (0-2): 2

Current board:
[['X' 'O' 'X']
 [None 'O' None]
 ['O' 'X' 'X']]

AI is thinking...

Current board:
[['X' 'O' 'X']
 [None 'O' 'O']
 ['O' 'X' 'X']]

Your turn!
Row (0-2): 1
Col (0-2): 0

Current board:
[['X' 'O' 'X']
 ['X' 'O' 'O']
 ['O' 'X' 'X']]
Game Over: Tie.


In [4]:
# ============================================
# TIC TAC TOE + MINIMAX + ALPHA-BETA PRUNING
# ============================================

import os
import math


# ---------------------------------------------------
# HÀM KIỂM TRA NGƯỜI THẮNG
# ---------------------------------------------------
def GetWinner(board):
    """
    Kiểm tra xem có người chơi nào thắng chưa.
    board là list 9 phần tử.
    Trả về "X", "O" hoặc None.
    """

    # Hàng ngang
    if board[0] == board[1] == board[2]:
        return board[0]
    elif board[3] == board[4] == board[5]:
        return board[3]
    elif board[6] == board[7] == board[8]:
        return board[6]

    # Cột dọc
    elif board[0] == board[3] == board[6]:
        return board[0]
    elif board[1] == board[4] == board[7]:
        return board[1]
    elif board[2] == board[5] == board[8]:
        return board[2]

    # Đường chéo
    elif board[0] == board[4] == board[8]:
        return board[0]
    elif board[2] == board[4] == board[6]:
        return board[2]

    return None


# ---------------------------------------------------
# IN BÀN CỜ
# ---------------------------------------------------
def PrintBoard(board):
    """
    In bàn cờ ra console.
    """
    os.system('cls' if os.name == 'nt' else 'clear')
    print(f"""
{board[0]} | {board[1]} | {board[2]}
{board[3]} | {board[4]} | {board[5]}
{board[6]} | {board[7]} | {board[8]}
""")


# ---------------------------------------------------
# TRẢ VỀ DANH SÁCH Ô TRỐNG
# ---------------------------------------------------
def GetAvailableCells(board):
    """
    Trả về danh sách ô chưa đánh.
    (ô là số nguyên 1–9)
    """
    return [cell for cell in board if cell != "X" and cell != "O"]


# ---------------------------------------------------
# MINIMAX + ALPHA-BETA PRUNING
# ---------------------------------------------------
def minimax(position, depth, alpha, beta, isMaximizing):
    """
    Thuật toán minimax với cắt tỉa alpha-beta.
    position: trạng thái hiện tại
    depth: độ sâu
    isMaximizing: True nếu là lượt "X", False nếu "O"
    """

    # Lấy kết quả hiện tại
    winner = GetWinner(position)
    if winner is not None:
        # X thắng → 10 - depth
        # O thắng → -10 + depth
        return 10 - depth if winner == "X" else -10 + depth

    # Trường hợp hòa
    if len(GetAvailableCells(position)) == 0:
        return 0

    # Trường hợp lượt của X (maximizing player)
    if isMaximizing:
        maxEval = -math.inf
        for cell in GetAvailableCells(position):
            position[cell - 1] = "X"
            Eval = minimax(position, depth + 1, alpha, beta, False)
            maxEval = max(maxEval, Eval)
            alpha = max(alpha, Eval)

            # Hoàn tác
            position[cell - 1] = cell

            # Cắt tỉa
            if beta <= alpha:
                break

        return maxEval

    # Trường hợp lượt của O (minimizing player)
    else:
        minEval = math.inf
        for cell in GetAvailableCells(position):
            position[cell - 1] = "O"
            Eval = minimax(position, depth + 1, alpha, beta, True)
            minEval = min(minEval, Eval)
            beta = min(beta, Eval)

            # Hoàn tác
            position[cell - 1] = cell

            # Cắt tỉa
            if beta <= alpha:
                break

        return minEval


# ---------------------------------------------------
# TÌM NƯỚC ĐI TỐI ƯU NHẤT
# ---------------------------------------------------
def FindBestMove(currentPosition, AI):
    """
    Trả về nước đi tốt nhất của AI.
    AI = "X" hoặc "O"
    """
    bestVal = -math.inf if AI == "X" else math.inf
    bestMove = -1

    for cell in GetAvailableCells(currentPosition):
        # Giả lập đánh thử
        currentPosition[cell - 1] = AI

        # X tối đa -> minimax() của lượt O
        # O tối thiểu -> minimax() của lượt X
        moveVal = minimax(
            currentPosition,
            0,
            -math.inf,
            math.inf,
            False if AI == "X" else True
        )

        # Hoàn tác
        currentPosition[cell - 1] = cell

        # Cập nhật nước đi tốt nhất
        if AI == "X" and moveVal > bestVal:
            bestMove = cell
            bestVal = moveVal
        elif AI == "O" and moveVal < bestVal:
            bestMove = cell
            bestVal = moveVal

    return bestMove


# ---------------------------------------------------
# GAME LOOP
# ---------------------------------------------------
def main():
    player = input("Play as X or O? ").strip().upper()
    AI = "O" if player == "X" else "X"

    # Bàn cờ ban đầu
    currentGame = list(range(1, 10))

    currentTurn = "X"   # X luôn đi trước
    moveCount = 0       # đếm số nước đi

    while True:
        # Lượt AI
        if currentTurn == AI:
            cell = FindBestMove(currentGame, AI)
            currentGame[cell - 1] = AI
            currentTurn = player

        # Lượt người chơi
        elif currentTurn == player:
            PrintBoard(currentGame)
            while True:
                human = int(input("Enter number (1–9): "))

                if human in currentGame:
                    currentGame[human - 1] = player
                    currentTurn = AI
                    break
                else:
                    PrintBoard(currentGame)
                    print("Cell not available!")

        # Kiểm tra thắng
        if GetWinner(currentGame) is not None:
            PrintBoard(currentGame)
            print(f"{GetWinner(currentGame)} WON!!!")
            break

        moveCount += 1

        # Hòa
        if GetWinner(currentGame) is None and moveCount == 9:
            PrintBoard(currentGame)
            print("TIE!")
            break


# ---------------------------------------------------
# CHẠY GAME
# ---------------------------------------------------
if __name__ == "__main__":
    main()


Play as X or O? X

1 | 2 | 3
4 | 5 | 6
7 | 8 | 9

Enter number (1–9): 1

X | 2 | 3
4 | O | 6
7 | 8 | 9

Enter number (1–9): 9

X | O | 3
4 | O | 6
7 | 8 | X

Enter number (1–9): 8

X | O | 3
4 | O | 6
O | X | X

Enter number (1–9): 3

X | O | X
4 | O | O
O | X | X

Enter number (1–9): 4

X | O | X
X | O | O
O | X | X

TIE!
